# Notebook 08 — Venue-Year Source Resolution via Award Paper OpenAlex IDs

## Motivation

Previous notebooks (05, 07, 07a, 07b) tried to resolve conference names → OpenAlex Source IDs
by **searching the `/sources` endpoint** with conference name strings. This produced two systematic failures:

1. **Wrong entity type:** OpenAlex returned `institution`, `repository`, or `journal` entities
   instead of `conference` proceedings. E.g. ACL resolved to "Association for Computational
   Linguistics" as an institution — not the ACL proceedings venue.
2. **170 / 448 (conf, year) pairs** remained unresolved even after multiple fix attempts (NB07b).
   Inspecting `nb07b_fix_venue_year_source_lookup.csv` shows only 41 of 278 resolved pairs
   have `source_type == 'conference'`. The rest resolved to repositories, journals, or ebook platforms.

## New Approach: Bottom-Up from Award Papers

Since we already have `openalex_id` for **903 / 925** award papers in `huang_matched_openalex.csv`,
we can **ask OpenAlex directly** what source each award paper belongs to by fetching the work record
and reading `primary_location.source`. This gives us the **ground-truth Source ID** that OpenAlex
actually uses for that paper — no guessing from conference name strings.

## API Authentication

This notebook uses your OpenAlex API key via an **environment variable** — the key is never
hardcoded in this file. Before running, set it in your terminal:

```bash
export OPENALEX_KEY="your_api_key_here"
```

Or add it to a `.env` file in the project root (make sure `.env` is in `.gitignore`):

```
OPENALEX_KEY=your_api_key_here
```

## Pipeline

**Step 1 — Extract source IDs from award paper work records**
- For each of the 903 award papers with an OpenAlex ID, fetch the work record
- Extract `primary_location.source.id`, `source.display_name`, `source.type`
- Group by `(conference, year)` → vote for the most common source_id per pair
- Output: `output/nb08_award_paper_source_ids.csv`

**Step 2 — Fetch ALL papers per (conf, year) using the resolved source IDs**
- For each resolved `(conference, year, source_id)`, call
  `GET /works?filter=primary_location.source.id:{source_id},publication_year:{year}`
- Paginate through all results (cursor-based, 200 per page)
- Exclude the award paper itself (by openalex_id) from the non-award pool
- Record: total papers found, award papers removed, non-award pool size
- Output: `output/nb08_nonaward_pool_counts.csv` (coverage summary)
- Output: `output/nb08_nonaward_works_raw.jsonl` (full work records, one per line)

**Step 3 — Coverage report**
- For every (conf, year) pair: resolved? source_type correct? how many non-award papers?
- Flag pairs with zero results or wrong source type for manual review
- Output: `output/nb08_coverage_report.csv`

## Inputs
- `../conf_data/huang_matched_openalex.csv` — 925 award papers with openalex_ids

## Outputs
- `output/nb08_award_paper_source_ids.csv` — per-(conf,year) source resolution
- `output/nb08_nonaward_pool_counts.csv` — count of non-award papers per (conf,year)
- `output/nb08_coverage_report.csv` — full coverage table for supervisor
- `output/nb08_nonaward_works_raw.jsonl` — raw work records (author info etc.) for downstream use
- `output/nb08_step1_cache.json` — API cache (avoid re-fetching on re-run)
- `output/nb08_step2_cache.json` — pagination cache per (conf,year)


## Setup

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
from pathlib import Path
from collections import Counter

# ---------------------------------------------------------------------------
# API key — read from environment variable, never hardcode here.
# Set before launching Jupyter:  export OPENALEX_KEY="your_key"
# Or load from a .env file (pip install python-dotenv):
#   from dotenv import load_dotenv; load_dotenv()
# ---------------------------------------------------------------------------
API_KEY  = os.environ.get("OPENALEX_KEY", "")
HEADERS  = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
MAILTO   = "sherotowshaw@gmail.com"   # polite-pool fallback (used even with key)

if API_KEY:
    print("API key loaded — using authenticated requests")
else:
    print("WARNING: OPENALEX_KEY not set — falling back to polite pool (slower)")

DATA_DIR  = Path('../conf_data')
OUT_DIR   = Path('output')
OUT_DIR.mkdir(exist_ok=True)

BASE_URL  = 'https://api.openalex.org'
SLEEP_S   = 0.07   # ~14 req/s — safe with API key; polite pool uses 0.12
PAGE_SIZE = 200    # max allowed by OpenAlex

df = pd.read_csv(DATA_DIR / 'huang_matched_openalex.csv')
print(f'Total award papers: {len(df)}')
print(f'With OpenAlex ID:   {df["openalex_id"].notna().sum()}')
print(f'Without OpenAlex ID:{df["openalex_id"].isna().sum()}')
print(f'Unique (conf,year) pairs: {df.groupby(["conference","year"]).ngroups}')


API key loaded — using authenticated requests
Total award papers: 925
With OpenAlex ID:   903
Without OpenAlex ID:22
Unique (conf,year) pairs: 453


## Step 1 — Fetch Source ID from Each Award Paper Work Record

For each award paper that has an `openalex_id`, we call `GET /works/{id}` and read
`primary_location.source`. We then vote per `(conference, year)` on the most common
source_id — this handles cases where a conf has multiple proceedings volumes in one year.

**Cache:** results stored in `output/nb08_step1_cache.json`. Re-running skips already-fetched IDs.

In [2]:
STEP1_CACHE = OUT_DIR / 'nb08_step1_cache.json'

# Reset cache once — the old cache used "first available location" logic,
# which favored repository/journal sources over conference sources.
# Delete it so Step 1 re-fetches with the new ranked-selection logic below.
if STEP1_CACHE.exists():
    STEP1_CACHE.unlink()
    print(f'Deleted stale cache: {STEP1_CACHE}')

step1_cache = {}
print('Cache reset — will re-fetch all work records with ranked source selection')

matched = df[df['openalex_id'].notna()].copy()
matched['work_id'] = matched['openalex_id'].str.rstrip('/').str.split('/').str[-1]

to_fetch = matched[~matched['work_id'].isin(step1_cache)]
print(f'Records to fetch this run: {len(to_fetch)}')


Deleted stale cache: output\nb08_step1_cache.json
Cache reset — will re-fetch all work records with ranked source selection
Records to fetch this run: 903


In [3]:
def source_rank(source):
    """Lower rank = more preferred venue type."""
    priority = {
        "conference": 0,
        "journal": 1,
        "book series": 2,
        "ebook platform": 3,
        "repository": 4,
    }
    return priority.get(source.get("type"), 99)


def fetch_work_source(work_id):
    """Fetch ALL candidate source locations for a work and keep the
    best-ranked one (conference > journal > book series > ebook > repository).
    Returns dict or None on 429."""
    try:
        r = requests.get(
            f'{BASE_URL}/works/{work_id}',
            params={'select': 'id,primary_location,locations', 'mailto': MAILTO},
            headers=HEADERS,
            timeout=20
        )
        if r.status_code == 429:
            time.sleep(5)
            return None
        if r.status_code == 404:
            return {'error': 'NOT_FOUND'}
        r.raise_for_status()
        data = r.json()

        candidates = []

        primary_source = (data.get('primary_location') or {}).get('source')
        if primary_source and primary_source.get('id'):
            candidates.append(('primary', primary_source))

        for loc in (data.get('locations') or []):
            s = loc.get('source')
            if s and s.get('id'):
                candidates.append(('location', s))

        if not candidates:
            return {'error': 'NO_SOURCE'}

        tier, best = min(candidates, key=lambda item: source_rank(item[1]))

        return {
            'source_id':   best['id'],
            'source_name': best.get('display_name'),
            'source_type': best.get('type'),
            'tier':        f'best_{tier}',
            'n_candidates': len(candidates)
        }

    except requests.exceptions.Timeout:
        return {'error': 'TIMEOUT'}
    except Exception as e:
        return {'error': f'EXCEPTION:{repr(e)}'}


failed = 0
for i, (_, row) in enumerate(to_fetch.iterrows()):
    wid = row['work_id']
    result = fetch_work_source(wid)
    if result is None:   # 429 — pause and retry once
        time.sleep(10)
        result = fetch_work_source(wid)
    if result is None:
        result = {'error': 'RATE_LIMIT_EXHAUSTED'}
    step1_cache[wid] = result
    if 'error' in result:
        failed += 1
    time.sleep(SLEEP_S)
    if (i + 1) % 50 == 0:
        with open(STEP1_CACHE, 'w') as f:
            json.dump(step1_cache, f)
        print(f'  Progress: {i+1}/{len(to_fetch)} | errors so far: {failed}')

with open(STEP1_CACHE, 'w') as f:
    json.dump(step1_cache, f)

print(f'\nDone. Total cached: {len(step1_cache)} | Total errors: {failed}')


  Progress: 50/903 | errors so far: 28
  Progress: 100/903 | errors so far: 44
  Progress: 150/903 | errors so far: 66
  Progress: 200/903 | errors so far: 84
  Progress: 250/903 | errors so far: 107
  Progress: 300/903 | errors so far: 125
  Progress: 350/903 | errors so far: 158
  Progress: 400/903 | errors so far: 176
  Progress: 450/903 | errors so far: 196
  Progress: 500/903 | errors so far: 227
  Progress: 550/903 | errors so far: 256
  Progress: 600/903 | errors so far: 286
  Progress: 650/903 | errors so far: 310
  Progress: 700/903 | errors so far: 336
  Progress: 750/903 | errors so far: 362
  Progress: 800/903 | errors so far: 383
  Progress: 850/903 | errors so far: 408
  Progress: 900/903 | errors so far: 433

Done. Total cached: 903 | Total errors: 433


In [4]:
# DIAGNOSTIC — run before cell 4
from collections import Counter

error_types = Counter()
for wid, result in step1_cache.items():
    if 'error' in result:
        error_types[result['error']] += 1

print(f"Total errors: {sum(error_types.values())}")
print("\nError breakdown:")
for err, count in error_types.most_common():
    print(f"  {err:40s}  {count}")

Total errors: 433

Error breakdown:
  NO_SOURCE                                 433


### Step 1 Results — Vote Per (Conference, Year)

For each (conf, year) we may have multiple award papers. We pick the most common `source_id`
across all award papers in that pair (majority vote). If all papers agree → high confidence.
If split → we flag for review.

In [5]:
matched['work_id'] = matched['openalex_id'].str.rstrip('/').str.split('/').str[-1]
matched['source_id']   = matched['work_id'].map(lambda w: step1_cache.get(w, {}).get('source_id'))
matched['source_name'] = matched['work_id'].map(lambda w: step1_cache.get(w, {}).get('source_name'))
matched['source_type'] = matched['work_id'].map(lambda w: step1_cache.get(w, {}).get('source_type'))
matched['fetch_error'] = matched['work_id'].map(lambda w: step1_cache.get(w, {}).get('error'))

print('Source type distribution across award papers:')
print(matched['source_type'].value_counts(dropna=False))
print(f'\nAward papers with source_id resolved: {matched["source_id"].notna().sum()} / {len(matched)}')
print(f'Award papers with no source: {matched["source_id"].isna().sum()}')

# Vote per (conference, year)
rows = []
for (conf, year), grp in matched.groupby(['conference', 'year']):
    resolved = grp[grp['source_id'].notna()]
    n_papers = len(grp)
    n_resolved = len(resolved)

    if n_resolved == 0:
        rows.append({
            'conference': conf, 'year': year,
            'source_id': None, 'source_name': None, 'source_type': None,
            'n_award_papers': n_papers, 'n_resolved': 0,
            'n_votes_for_winner': 0, 'vote_agreement': None,
            'status': 'UNRESOLVED'
        })
        continue

    counts = Counter(resolved['source_id'])
    winner_id, winner_votes = counts.most_common(1)[0]
    winner_row = resolved[resolved['source_id'] == winner_id].iloc[0]

    rows.append({
        'conference': conf, 'year': year,
        'source_id':   winner_id,
        'source_name': winner_row['source_name'],
        'source_type': winner_row['source_type'],
        'n_award_papers': n_papers,
        'n_resolved': n_resolved,
        'n_votes_for_winner': winner_votes,
        'vote_agreement': round(winner_votes / n_resolved, 2),
        'status': 'RESOLVED'
    })

source_df = pd.DataFrame(rows).sort_values(['conference', 'year']).reset_index(drop=True)
source_df.to_csv(OUT_DIR / 'nb08_award_paper_source_ids.csv', index=False)

print(f'\n(conf, year) pairs total:    {len(source_df)}')
print(f'Resolved:                    {(source_df["status"]=="RESOLVED").sum()}')
print(f'Unresolved:                  {(source_df["status"]=="UNRESOLVED").sum()}')
print(f'\nSource type breakdown (resolved pairs):')
print(source_df[source_df['status']=='RESOLVED']['source_type'].value_counts())
source_df.head(15)


Source type distribution across award papers:
source_type
None              433
repository        226
journal           149
conference         86
ebook platform      8
book series         1
Name: count, dtype: int64

Award papers with source_id resolved: 470 / 903
Award papers with no source: 433

(conf, year) pairs total:    448
Resolved:                    284
Unresolved:                  164

Source type breakdown (resolved pairs):
source_type
repository        130
journal           103
conference         47
ebook platform      3
book series         1
Name: count, dtype: int64


,conference,year,source_id,source_name,source_type,n_award_papers,n_resolved,n_votes_for_winner,vote_agreement,status
0,AAAI,2000,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,4,4,4,1.0,RESOLVED
1,AAAI,2002,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,1,1,1,1.0,RESOLVED
2,AAAI,2004,https://openalex.org/S4210169993,Civil War Book Review,journal,1,1,1,1.0,RESOLVED
3,AAAI,2005,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,1,1,1,1.0,RESOLVED
4,AAAI,2006,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,2,2,2,1.0,RESOLVED
5,AAAI,2007,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,2,2,2,1.0,RESOLVED
6,AAAI,2008,https://openalex.org/S4306420577,National Conference on Artificial Intelligence,conference,2,2,2,1.0,RESOLVED
7,AAAI,2010,https://openalex.org/S4210191458,Proceedings of the AAAI Conference on Artifici...,conference,2,2,1,0.5,RESOLVED
8,AAAI,2011,https://openalex.org/S4210191458,Proceedings of the AAAI Conference on Artifici...,conference,2,2,2,1.0,RESOLVED
9,AAAI,2012,https://openalex.org/S4210191458,Proceedings of the AAAI Conference on Artifici...,conference,2,2,2,1.0,RESOLVED


In [6]:
# Spot-check: what are the top repository/journal source names?
check = source_df[source_df['status'] == 'RESOLVED'].copy()
print("=== TOP REPOSITORY SOURCES ===")
print(check[check['source_type']=='repository']['source_name'].value_counts().head(20))
print("\n=== TOP JOURNAL SOURCES ===")
print(check[check['source_type']=='journal']['source_name'].value_counts().head(20))
print("\n=== CONFERENCE SOURCES ===")
print(check[check['source_type']=='conference']['source_name'].value_counts().head(20))

=== TOP REPOSITORY SOURCES ===
source_name
arXiv (Cornell University)                                                                                                                     25
DSpace@MIT (Massachusetts Institute of Technology)                                                                                             10
HAL (Le Centre pour la Communication Scientifique Directe)                                                                                      8
Infoscience (Ecole Polytechnique Fédérale de Lausanne)                                                                                          5
UvA-DARE (University of Amsterdam)                                                                                                              3
UCL Discovery (University College London)                                                                                                       3
National University of Singapore                                                 

In [7]:
# Check: for papers where primary_location gave repo/journal,
# does the locations fallback have a conference-type source?
conf_from_fallback = 0
for wid, result in step1_cache.items():
    if result.get('tier') == 'fallback_location' and result.get('source_type') == 'conference':
        conf_from_fallback += 1

print(f"Conference sources found via fallback locations: {conf_from_fallback}")

# Also check what tiers the 470 resolved papers used
tier_counts = Counter(v.get('tier') for v in step1_cache.values() if 'source_id' in v)
print(f"\nResolution tier breakdown:")
for tier, count in tier_counts.most_common():
    print(f"  {tier}: {count}")

Conference sources found via fallback locations: 0

Resolution tier breakdown:
  best_location: 263
  best_primary: 207


## Step 2 — Fetch All Non-Award Papers Per (Conference, Year)

For every resolved (conf, year, source_id), we query:
```
GET /works?filter=primary_location.source.id:{source_id},publication_year:{year}&per-page=200
```
We paginate using cursor (`cursor=*` to start, then `meta.next_cursor`).
We remove award papers from the results (by openalex_id).

**Cache:** each (conf|year) key stored in `output/nb08_step2_cache.json` once fully fetched.
Re-running skips already-completed pairs.

> ⚠️ This step makes many API calls. With ~280 resolved pairs and ~200 papers per page,
> expect 300–800 API calls total (~1–2 minutes with API key).

In [ ]:
STEP2_CACHE  = OUT_DIR / 'nb08_step2_cache.json'
WORKS_JSONL  = OUT_DIR / 'nb08_nonaward_works_raw.jsonl'

if STEP2_CACHE.exists():
    with open(STEP2_CACHE) as f:
        step2_cache = json.load(f)
    print(f'Cache hit: {len(step2_cache)} (conf,year) pairs already fetched')
else:
    step2_cache = {}
    print('No step2 cache — starting fresh')

award_ids = set(df['openalex_id'].dropna().str.rstrip('/').str.split('/').str[-1])
print(f'Award paper IDs to exclude from pool: {len(award_ids)}')

resolved_pairs = source_df[source_df['status'] == 'RESOLVED'].copy()
to_process = resolved_pairs[
    ~resolved_pairs.apply(lambda r: f"{r['conference']}|{r['year']}", axis=1).isin(step2_cache)
]
print(f'Pairs to fetch this run: {len(to_process)}')


In [ ]:
def fetch_all_works_for_source_year(source_id, year, conf, award_ids_set):
    """
    Paginate through all works for a given source_id + year.
    Returns dict with: works (list of slim records), total_fetched, nonaward_count, errors.
    """
    short_id = source_id.rstrip('/').split('/')[-1]
    base_filter = f'primary_location.source.id:{short_id},publication_year:{year}'
    select_fields = 'id,title,publication_year,authorships,cited_by_count,doi'

    all_works = []
    cursor = '*'
    errors = []
    page = 0

    while cursor:
        params = {
            'filter':   base_filter,
            'per-page': PAGE_SIZE,
            'cursor':   cursor,
            'select':   select_fields,
            'mailto':   MAILTO,
        }
        try:
            r = requests.get(
                f'{BASE_URL}/works',
                params=params,
                headers=HEADERS,
                timeout=30
            )
            if r.status_code == 429:
                time.sleep(10)
                continue   # retry same cursor
            r.raise_for_status()
            data = r.json()
        except requests.exceptions.Timeout:
            errors.append(f'TIMEOUT_page{page}')
            break
        except Exception as e:
            errors.append(f'ERROR_page{page}:{repr(e)}')
            break

        results = data.get('results', [])
        meta    = data.get('meta', {})

        for w in results:
            wid = w.get('id', '').rstrip('/').split('/')[-1]
            all_works.append({
                'work_openalex_id': w.get('id'),
                'title':            w.get('title'),
                'publication_year': w.get('publication_year'),
                'doi':              w.get('doi'),
                'cited_by_count':   w.get('cited_by_count'),
                'authorships':      w.get('authorships'),
                'conference':       conf,
                'year':             year,
                'source_id':        source_id,
                'is_award':         wid in award_ids_set
            })

        cursor = meta.get('next_cursor')  # None when last page
        page  += 1
        time.sleep(SLEEP_S)

        if not results:
            break

    nonaward = [w for w in all_works if not w['is_award']]
    return {
        'total_fetched':   len(all_works),
        'award_found':     len(all_works) - len(nonaward),
        'nonaward_count':  len(nonaward),
        'pages':           page,
        'errors':          errors,
        'works':           nonaward
    }


jsonl_mode = 'a' if STEP2_CACHE.exists() else 'w'

with open(WORKS_JSONL, jsonl_mode) as jsonl_f:
    for i, (_, row) in enumerate(to_process.iterrows()):
        key = f"{row['conference']}|{row['year']}"
        result = fetch_all_works_for_source_year(
            row['source_id'], row['year'], row['conference'], award_ids
        )

        for w in result['works']:
            jsonl_f.write(json.dumps(w) + '\n')

        step2_cache[key] = {
            'total_fetched':  result['total_fetched'],
            'award_found':    result['award_found'],
            'nonaward_count': result['nonaward_count'],
            'pages':          result['pages'],
            'errors':         result['errors'],
            'status':         'OK' if not result['errors'] else 'PARTIAL'
        }

        with open(STEP2_CACHE, 'w') as f:
            json.dump(step2_cache, f)

        if (i + 1) % 10 == 0 or (i + 1) == len(to_process):
            print(f'  [{i+1}/{len(to_process)}] {key}: '
                  f'{result["nonaward_count"]} non-award papers | '
                  f'errors: {result["errors"]}')

print(f'\nStep 2 complete. Total (conf,year) pairs cached: {len(step2_cache)}')


## Step 3 — Coverage Report

Build the full supervisor-ready table:
- Every (conf, year) pair from the original 448
- Whether source resolved, source type, how many non-award papers found
- Flag: enough data (>= 10 non-award papers)? zero results? unresolved?

In [ ]:
all_pairs = df.groupby(['conference','year']).size().reset_index(name='n_award_papers')

report_rows = []
for _, row in all_pairs.iterrows():
    conf, year = row['conference'], row['year']
    key = f'{conf}|{year}'

    src_row = source_df[(source_df['conference']==conf) & (source_df['year']==year)]
    step2   = step2_cache.get(key, {})

    if src_row.empty or src_row.iloc[0]['status'] == 'UNRESOLVED':
        resolution_status = 'UNRESOLVED'
        source_id = source_type = source_name = None
        nonaward_count = total_fetched = 0
        step2_status = 'SKIPPED'
        flag = 'NO_SOURCE_ID'
    else:
        s = src_row.iloc[0]
        resolution_status = 'RESOLVED'
        source_id   = s['source_id']
        source_name = s['source_name']
        source_type = s['source_type']
        nonaward_count = step2.get('nonaward_count', 0)
        total_fetched  = step2.get('total_fetched', 0)
        step2_status   = step2.get('status', 'NOT_RUN')

        if nonaward_count == 0:
            flag = 'ZERO_RESULTS'
        elif nonaward_count < 5:
            flag = 'VERY_FEW_RESULTS (<5)'
        elif nonaward_count < 10:
            flag = 'FEW_RESULTS (<10)'
        else:
            flag = 'OK'

    report_rows.append({
        'conference':        conf,
        'year':              year,
        'n_award_papers':    row['n_award_papers'],
        'resolution_status': resolution_status,
        'source_id':         source_id,
        'source_name':       source_name,
        'source_type':       source_type,
        'total_papers_found':total_fetched,
        'nonaward_pool_size':nonaward_count,
        'step2_status':      step2_status,
        'flag':              flag if resolution_status == 'RESOLVED' else 'NO_SOURCE_ID'
    })

report_df = pd.DataFrame(report_rows).sort_values(['conference','year']).reset_index(drop=True)
report_df.to_csv(OUT_DIR / 'nb08_coverage_report.csv', index=False)

print('=== COVERAGE SUMMARY ===')
print(f'Total (conf, year) pairs:      {len(report_df)}')
print(f'Resolved:                      {(report_df["resolution_status"]=="RESOLVED").sum()}')
print(f'Unresolved (no source ID):     {(report_df["resolution_status"]=="UNRESOLVED").sum()}')
print()
print('Flag breakdown (resolved pairs):')
print(report_df[report_df['resolution_status']=='RESOLVED']['flag'].value_counts())
print()
print('Source type breakdown (resolved):')
print(report_df[report_df['resolution_status']=='RESOLVED']['source_type'].value_counts(dropna=False))
print()
print('Non-award pool stats (OK pairs only):')
ok = report_df[report_df['flag']=='OK']['nonaward_pool_size']
print(ok.describe().round(1))


## Step 4 — Per-Conference Summary Table (for Supervisor)

Aggregate to conference level showing: how many years resolved, how many years with enough data,
total non-award pool size available.

In [ ]:
conf_summary = report_df.groupby('conference').agg(
    total_years         = ('year', 'count'),
    years_resolved      = ('resolution_status', lambda x: (x=='RESOLVED').sum()),
    years_with_ok_data  = ('flag', lambda x: (x=='OK').sum()),
    total_nonaward_pool = ('nonaward_pool_size', 'sum'),
    avg_pool_per_year   = ('nonaward_pool_size', lambda x: round(x[x>0].mean(), 0) if (x>0).any() else 0)
).reset_index()

conf_summary['pct_resolved'] = (conf_summary['years_resolved'] / conf_summary['total_years'] * 100).round(1)
conf_summary = conf_summary.sort_values('years_resolved', ascending=False)

print('Per-conference resolution summary:')
print(conf_summary.to_string(index=False))

conf_summary.to_csv(OUT_DIR / 'nb08_conf_summary.csv', index=False)


## Step 5 — Inspect Unresolved & Zero-Result Pairs

Print a clean list of what failed and why — this is the reportable limitations section.

In [ ]:
print('=== UNRESOLVED (no source_id found from award papers) ===')
unresolved = report_df[report_df['resolution_status']=='UNRESOLVED']
print(f'Count: {len(unresolved)}')
print(unresolved[['conference','year','n_award_papers']].to_string(index=False))

print('\n=== ZERO RESULTS (source resolved but 0 papers returned) ===')
zero = report_df[report_df['flag']=='ZERO_RESULTS']
print(f'Count: {len(zero)}')
print(zero[['conference','year','source_id','source_type','total_papers_found']].to_string(index=False))

print('\n=== FEW RESULTS (1-9 non-award papers) ===')
few = report_df[report_df['flag'].isin(['VERY_FEW_RESULTS (<5)', 'FEW_RESULTS (<10)'])]
print(f'Count: {len(few)}')
print(few[['conference','year','source_type','nonaward_pool_size']].to_string(index=False))


## Summary & Next Steps

### What This Notebook Establishes
- A **ground-truth source ID** per (conf, year) pair, derived directly from OpenAlex work records
  of award papers — not inferred from conference name searches (which systematically resolved
  to wrong entity types: institutions, repositories, journals)
- A **count of all non-award papers** available per (conf, year) for use as control pool
- A **coverage report** documenting exactly which pairs have sufficient data and which do not

### Reportable to Supervisor
- Previous approach (NB05–07b) resolved only 41/448 pairs to `conference` type;
  237 resolved to wrong types (repository/journal) and 170 returned nothing
- This approach gets source IDs **directly from the award papers themselves**, which is
  the most reliable way to identify the correct OpenAlex venue entity
- Remaining unresolved pairs = award papers where OpenAlex has **no structured source metadata**
  at all in `primary_location` — a known OpenAlex data gap for older conference proceedings

### Next Step (Notebook 09)
Use `nb08_coverage_report.csv` + `nb08_nonaward_works_raw.jsonl` to:
1. Build author profiles for all non-award paper authors
2. Compute career ages and label Junior (≤5 years) / Senior (≥6 years)
3. Compare junior awardee trajectories vs. ALL junior non-awardees in same (conf, year)
